In [16]:
import requests
import time
from bs4 import BeautifulSoup
import sqlite3
import re

def create_database():
    """データベースとテーブルの作成"""
    conn = sqlite3.connect('github_repos.db')
    cursor = conn.cursor()
    
    # テーブルが存在しなければ作成
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS google_repos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        repo_name TEXT UNIQUE,
        main_language TEXT,
        stars INTEGER
    )
    ''')
    
    conn.commit()
    return conn, cursor

def scrape_google_repos(pages=1800):
    """Googleのリポジトリ情報をスクレイピング"""
    conn, cursor = create_database()
    
    for page in range(1, pages + 1):
        url = f'https://github.com/google?page={page}&tab=repositories'
        
        # スクレイピング
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            repo_list = soup.find_all('li', {'class': 'Box-row'})
            
            for repo in repo_list:
                # リポジトリ名取得
                repo_name_element = repo.find('a', itemprop='name codeRepository')
                if repo_name_element:
                    repo_name = repo_name_element.text.strip()
                else:
                    continue
                
                # 主要言語取得
                language_element = repo.find('span', itemprop='programmingLanguage')
                main_language = language_element.text.strip() if language_element else 'None'
                
                # スター数取得
                stars_element = repo.find('a', href=re.compile(f"/google/{repo_name}/stargazers"))
                stars_text = stars_element.text.strip() if stars_element else '0'
                stars = int(stars_text.replace(',', '')) if stars_text != '' else 0
                
                # データベースに保存
                try:
                    cursor.execute(
                        "INSERT OR REPLACE INTO google_repos (repo_name, main_language, stars) VALUES (?, ?, ?)",
                        (repo_name, main_language, stars)
                    )
                    conn.commit()
                    print(f"Saved: {repo_name}, {main_language}, {stars} stars")
                except Exception as e:
                    print(f"Error saving {repo_name}: {e}")
            
            print(f"Page {page} completed")
            
            # Rate limit対策
            time.sleep(1)
        else:
            print(f"Failed to retrieve page {page}: Status code {response.status_code}")
    
    conn.close()

def display_repos():
    """保存したリポジトリ情報を表示"""
    conn = sqlite3.connect('github_repos.db')
    cursor = conn.cursor()
    
    cursor.execute("SELECT repo_name, main_language, stars FROM google_repos ORDER BY stars DESC")
    repos = cursor.fetchall()
    
    print("\n== Googleのリポジトリ情報 ==")
    print("リポジトリ名\t\t主要言語\t\tスター数")
    print("-" * 70)
    
    for repo in repos:
        repo_name, language, stars = repo
        print(f"{repo_name[:25]:<25} {language[:15]:<15} {stars:>10}")
    
    print(f"\n合計: {len(repos)}件のリポジトリ")
    conn.close()

if __name__ == "__main__":
    scrape_google_repos()
    display_repos()

Saved: perfetto, C++, 5000 stars
Saved: docsy-example, HTML, 537 stars
Saved: sedpack, Python, 28 stars
Saved: deps.dev, Go, 354 stars
Saved: site-kit-wp, JavaScript, 1337 stars
Saved: nomulus, Java, 1767 stars
Saved: docsy, JavaScript, 2865 stars
Saved: budoux, Python, 1548 stars
Saved: or-tools, C++, 12713 stars
Saved: googletest-rust, Rust, 393 stars
Page 1 completed
Saved: perfetto, C++, 5000 stars
Saved: docsy-example, HTML, 537 stars
Saved: sedpack, Python, 28 stars
Saved: deps.dev, Go, 354 stars
Saved: site-kit-wp, JavaScript, 1337 stars
Saved: nomulus, Java, 1767 stars
Saved: docsy, JavaScript, 2865 stars
Saved: budoux, Python, 1548 stars
Saved: or-tools, C++, 12713 stars
Saved: googletest-rust, Rust, 393 stars
Page 2 completed
Saved: perfetto, C++, 5000 stars
Saved: docsy-example, HTML, 537 stars
Saved: sedpack, Python, 28 stars
Saved: deps.dev, Go, 354 stars
Saved: site-kit-wp, JavaScript, 1337 stars
Saved: nomulus, Java, 1767 stars
Saved: docsy, JavaScript, 2865 stars
Saved: